<a href="https://colab.research.google.com/github/hanssingh79/Agentic-AI/blob/main/CRMAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import os
import json
from google.colab import userdata

In [13]:
# --- 1. Initialize Claude Client ---
from anthropic import Anthropic
try:
    client = Anthropic(api_key=userdata.get('CLAUDE_API_KEY'))
except Exception as e:
    print(f"Error initializing Claude client: {e}")
    print("Please ensure your CLAUDE_API_KEY is set in your environment variables.")

In [14]:

def lookup_domain_info(domain: str) -> str:
    """
    Looks up and returns mock company information based on its domain.
    In a real system, this would call an external data enrichment API (e.g., Clearbit).
    """
    print(f"-> TOOL ACTIVATED: Looking up domain info for {domain}...")

    # Mock database for demonstration
    mock_data = {
        "acmecorp.com": {"industry": "Software/SaaS", "size": "501-1000 employees", "revenue": "$50M - $100M"},
        "widgetco.net": {"industry": "Manufacturing", "size": "100-250 employees", "revenue": "$10M - $25M"},
        "globalfin.org": {"industry": "Financial Services", "size": "5000+ employees", "revenue": "$1B+"},
    }

    info = mock_data.get(domain, {"industry": "Unknown", "size": "N/A", "revenue": "N/A"})

    # Return the data as a JSON string for the AI model to parse easily
    return json.dumps(info)

In [15]:
def check_crm_history(email: str) -> str:
    """
    Checks the internal CRM system for past engagement history with the lead.
    In a real system, this would query a PostgreSQL database or a CRM API (e.g., Salesforce).
    """
    print(f"-> TOOL ACTIVATED: Checking CRM history for {email}...")

    # Mock database for demonstration
    mock_data = {
        "jane@acmecorp.com": {"last_contact": "2025-11-15", "status": "Cold Lead", "notes": "Attended webinar, no follow-up yet."},
        "bob@widgetco.net": {"last_contact": "2025-12-01", "status": "Active Opportunity", "notes": "Discussed Q1 budget and product integration."},
        "default": {"last_contact": "N/A", "status": "No Record", "notes": "New lead, first contact opportunity."},
    }

    history = mock_data.get(email, mock_data["default"])
    return json.dumps(history)


In [16]:
def calculate_lead_score(data_summary: str) -> str:
    """
    Analyzes the collected data (domain and CRM history) to assign a lead score (High/Medium/Low).
    This function simulates a complex scoring algorithm.
    """
    print("-> TOOL ACTIVATED: Calculating lead score...")

    data = json.loads(data_summary)
    score = "Low" # Default score

    # Simple scoring logic for demonstration
    if data["domain_info"].get("revenue", "").startswith("$1B+"):
        score = "High"
    elif data["crm_history"].get("status") == "Active Opportunity":
        score = "High"
    elif data["domain_info"].get("revenue", "").startswith("$50M"):
        score = "Medium"

    return json.dumps({"lead_score": score})

In [17]:
# Map of available function names to the actual Python functions
AVAILABLE_FUNCTIONS = {
    "lookup_domain_info": lookup_domain_info,
    "check_crm_history": check_crm_history,
    "calculate_lead_score": calculate_lead_score,
}

In [18]:
# This is how the AI model learns about the tools it can use.
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "lookup_domain_info",
            "description": "Retrieves general business information (industry, size, revenue) about a company based on its domain name.",
            "parameters": {
                "type": "object",
                "properties": {
                    "domain": {"type": "string", "description": "The company's domain name, e.g., 'acmecorp.com'"},
                },
                "required": ["domain"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "check_crm_history",
            "description": "Checks the internal CRM system for past contact, status, and notes associated with a specific lead email.",
            "parameters": {
                "type": "object",
                "properties": {
                    "email": {"type": "string", "description": "The full email address of the lead."},
                },
                "required": ["email"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_lead_score",
            "description": "Calculates the priority score (High/Medium/Low) for a lead based on a summary of all collected domain and CRM history data.",
            "parameters": {
                "type": "object",
                "properties": {
                    "data_summary": {"type": "string", "description": "A JSON string containing the combined domain_info and crm_history."},
                },
                "required": ["data_summary"],
            },
        },
    },
]

In [21]:
def run_agent(user_prompt: str):
    """
    The main execution loop for the CRM Lead Qualifier Agent.
    """
    print(f"\n--- Running Lead Qualifier Agent ---")

    system_prompt = (
        "You are an expert CRM Lead Qualifier Agent. Your sole task is to analyze a sales lead "
        "provided via email address. You must follow these steps precisely: "
        "1. Identify the domain from the email. "
        "2. Call `lookup_domain_info` and `check_crm_history` sequentially to gather all data. "
        "3. Combine all collected data into a single JSON object. "
        "4. Call `calculate_lead_score` with the combined JSON object. "
        "5. Finally, synthesize all information (domain info, CRM history, and score) "
        "into a single, easy-to-read summary for a busy sales rep."
    )

    messages = [
        {"role": "user", "content": user_prompt},
    ]

    collected_data = {}

    while True:
        print("\n[AI Thinking...]")

        # Prepare messages for Anthropic's client.messages.create
        # Anthropic's system prompt is passed separately.

        # Extract tool_choice if present from the last message
        anthropic_messages = []
        for msg in messages:
            if msg["role"] == "user" or msg["role"] == "assistant":
                anthropic_messages.append(msg)
            elif msg["role"] == "tool":
                # Anthropic tool messages structure differently
                anthropic_messages.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": msg["tool_call_id"], "content": msg["content"]}]})

        response = client.messages.create(
            model="claude-3-opus-20240229", # Using a Claude model
            max_tokens=2000,
            system=system_prompt,
            messages=anthropic_messages,
            tools=tools_schema,
            tool_choice={"type": "auto"}
        )

        response_message = response.content[0]

        if response.stop_reason == "tool_use":
            tool_call = response_message
            function_name = tool_call.name
            function_to_call = AVAILABLE_FUNCTIONS.get(function_name)
            function_args = tool_call.input

            if not function_to_call:
                print(f"Error: Unknown function {function_name}")
                # Handle gracefully, maybe add an error message to messages
                break

            # Execute the function
            function_result = function_to_call(**function_args)

            # Update the persistent memory (collected_data)
            if function_name == "lookup_domain_info":
                collected_data["domain_info"] = json.loads(function_result)
            elif function_name == "check_crm_history":
                collected_data["crm_history"] = json.loads(function_result)
            elif function_name == "calculate_lead_score":
                # Inject the accumulated data from previous turns
                # The tool_call.input for calculate_lead_score expects data_summary
                # So we update function_args before calling.
                function_args = {"data_summary": json.dumps(collected_data)}
                function_result = function_to_call(**function_args)

            # Append the tool result as a NEW message for Anthropic
            messages.append({
                "role": "assistant",
                "content": [tool_call]
            })
            messages.append({
                "role": "user",
                "content": [
                    {
                        "type": "tool_result",
                        "tool_use_id": tool_call.id,
                        "content": function_result,
                    }
                ],
            })

        else:
            print("\n--- FINAL AGENT SUMMARY ---")
            print(response_message.text)
            messages.append({"role": "assistant", "content": response_message.text})
            break

In [22]:
# Scenario 1: High-Value Lead (Large company, needs scoring)
lead_email_1 = "jane@acmecorp.com"
run_agent(f"Please qualify this lead for my call tomorrow: {lead_email_1}")

print("\n" + "="*80 + "\n")


--- Running Lead Qualifier Agent ---

[AI Thinking...]


/tmp/ipykernel_322/3373140641.py:39: DeprecationWarning: The model 'claude-3-opus-20240229' is deprecated and will reach end-of-life on January 5th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  response = client.messages.create(


BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011CZ6XjbsiVJ2rVTSKmeKXf'}